# HW3: Russian NER on factRuEval-2016

Этот ноутбук не переобучает модели, а читает артефакты из `artifacts_hw3/` и собирает итоговую сводку по экспериментам.

## Imports

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

ARTIFACTS_DIR = Path('artifacts_hw3')

## Dataset

In [2]:
dataset_info_path = ARTIFACTS_DIR / 'dataset_inspect' / 'dataset_info.json'
dataset_info = json.loads(dataset_info_path.read_text(encoding='utf-8'))
dataset_info

{'dataset_config_name': None,
 'dataset_name': 'gusevski/factrueval2016',
 'label_column': 'ner_tags',
 'label_names': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'],
 'splits': {'test': 2582, 'train': 7746, 'validation': 2582},
 'token_column': 'tokens'}

## Experiment Summary

In [3]:
summary_path = ARTIFACTS_DIR / 'run_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))
else:
    print('run_summary.md not found yet')

| run_name | command | model_name_or_path | test_before_f1 | test_after_f1 | delta_f1 | test_after_precision | test_after_recall | eval_loss | epoch | eval_runtime | eval_samples_per_second | eval_steps_per_second |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ner_from_entity_mlm_with_synthetic_titles_10k | train-ner | artifacts_hw3/mlm_rubert_tiny2_entity_e1/checkpoint-best | 0.0326 | 0.8328 | 0.8002 | 0.7972 | 0.8717 |  |  |  |  |  |
| ner_with_synthetic_titles_10k | train-ner | cointegrated/rubert-tiny2 | 0.0318 | 0.8251 | 0.7933 | 0.7856 | 0.8688 |  |  |  |  |  |
| ner_from_mlm_rubert_tiny2_entity_e1 | train-ner | artifacts_hw3/mlm_rubert_tiny2_entity_e1/checkpoint-best | 0.0326 | 0.7721 | 0.7394 | 0.7372 | 0.8105 |  |  |  |  |  |
| ner_from_mlm_rubert_tiny2_whole_word_e1 | train-ner | artifacts_hw3/mlm_rubert_tiny2_whole_word_e1/checkpoint-best | 0.0322 | 0.7487 | 0.7165 | 0.7102 | 0.7916 |  |  |  |  |  |
| baseline_rubert_tiny2_e1 | train-ner | cointegrated/rubert-tiny2 | 0.0318 | 0.7367 | 0.7050 | 0.6953 | 0.7834 |  |  |  |  |  |
| smoke_ner_tiny_v2 | train-ner | hf-internal-testing/tiny-random-bert | 0.0414 | 0.0420 | 0.0007 | 0.0244 | 0.1519 |  |  |  |  |  |
| mlm_rubert_tiny2_entity_e1 | train-mlm | cointegrated/rubert-tiny2 |  |  |  |  |  | 4.6614 | 1.0000 | 53.5351 | 48.2300 | 1.5130 |
| mlm_rubert_tiny2_whole_word_e1 | train-mlm | cointegrated/rubert-tiny2 |  |  |  |  |  | 3.7948 | 1.0000 | 41.1995 | 62.6710 | 1.9660 |
| smoke_mlm_entity | train-mlm | hf-internal-testing/tiny-random-bert |  |  |  |  |  | 7.0099 | 1.0000 | 0.0698 | 458.1700 | 57.2710 |
| smoke_mlm_whole_word | train-mlm | hf-internal-testing/tiny-random-bert |  |  |  |  |  | 7.0203 | 1.0000 | 0.0769 | 416.0060 | 52.0010 |

In [4]:
rows = []
for metrics_path in sorted(ARTIFACTS_DIR.glob('*/metrics.json')):
    run_dir = metrics_path.parent
    run_name = run_dir.name
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    run_config_path = run_dir / 'run_config.json'
    run_config = json.loads(run_config_path.read_text(encoding='utf-8')) if run_config_path.exists() else {}

    before_metrics = metrics.get('before_finetuning', {})
    after_metrics = metrics.get('after_finetuning', {})
    rows.append({
        'run_name': run_name,
        'command': run_config.get('command'),
        'model_name_or_path': run_config.get('model_name_or_path'),
        'test_before_f1': before_metrics.get('test_before_f1'),
        'test_after_f1': after_metrics.get('test_after_f1'),
        'delta_f1': None if before_metrics.get('test_before_f1') is None or after_metrics.get('test_after_f1') is None else after_metrics.get('test_after_f1') - before_metrics.get('test_before_f1'),
        'test_after_precision': after_metrics.get('test_after_precision'),
        'test_after_recall': after_metrics.get('test_after_recall'),
    })

pd.DataFrame(rows).sort_values('test_after_f1', ascending=False)

,run_name,command,model_name_or_path,test_before_f1,test_after_f1,delta_f1,test_after_precision,test_after_recall
3,ner_from_entity_mlm_with_synthetic_titles_10k,train-ner,artifacts_hw3/mlm_rubert_tiny2_entity_e1/check...,0.032641,0.832795,0.800154,0.797225,0.871687
6,ner_with_synthetic_titles_10k,train-ner,cointegrated/rubert-tiny2,0.031770,0.825102,0.793332,0.785620,0.868763
4,ner_from_mlm_rubert_tiny2_entity_e1,train-ner,artifacts_hw3/mlm_rubert_tiny2_entity_e1/check...,0.032641,0.772070,0.739429,0.737157,0.810455
5,ner_from_mlm_rubert_tiny2_whole_word_e1,train-ner,artifacts_hw3/mlm_rubert_tiny2_whole_word_e1/c...,0.032231,0.748725,0.716494,0.710233,0.791629
0,baseline_rubert_tiny2_e1,train-ner,cointegrated/rubert-tiny2,0.031770,0.736743,0.704973,0.695328,0.783403
9,smoke_ner_tiny_v2,train-ner,hf-internal-testing/tiny-random-bert,0.041379,0.042032,0.000652,0.024390,0.151899
1,mlm_rubert_tiny2_entity_e1,train-mlm,cointegrated/rubert-tiny2,NaN,NaN,NaN,NaN,NaN
2,mlm_rubert_tiny2_whole_word_e1,train-mlm,cointegrated/rubert-tiny2,NaN,NaN,NaN,NaN,NaN
7,smoke_mlm_entity,train-mlm,hf-internal-testing/tiny-random-bert,NaN,NaN,NaN,NaN,NaN
8,smoke_mlm_whole_word,train-mlm,hf-internal-testing/tiny-random-bert,NaN,NaN,NaN,NaN,NaN


## Чеклист по требованиям

- Загрузил `gusevski/factrueval2016`: включая разворачивание нестандартного вложенного поля `data`.
- Подготовка датасета для NER: выполнена, есть токенизация, выравнивание меток и нормализация схемы BIO.
- Baseline NER с замером метрик до и после fine-tuning: выполнен.
- Предварительное `MLM -> NER`: выполнено.
- Нестандартное маскирование для MLM: выполнено в двух вариантах, `whole_word` и `entity masking`.
- Синтетическая разметка внешнего корпуса: выполнена на `10_000` заголовках `Lenta.ru`, сохранено `7_761` pseudo-labeled примеров.
- Сравнение подходов: выполнено, сводка лежит в `artifacts_hw3/run_summary.md`.
- Воспроизводимость: фиксирован `seed=42`, все прогоны сохраняют `metrics.json` и `run_config.json`.
- Обоснованность решений: частично закрыта через `README` и артефакты экспериментов; в финальном тексте отчёта стоит явно проговорить ограничения `cpu-only` среды и выбор `rubert-tiny2` как компромисс между качеством и временем.

## Conclusions

- Лучший baseline в текущих экспериментах: `cointegrated/rubert-tiny2`, fine-tuning только на gold-разметке дал `test_after_f1 = 0.7367`. Это разумный базовый уровень для `cpu-only` среды.
- Ветка `MLM -> NER` действительно помогает. `whole_word masking` улучшил качество до `0.7487`, а `entity masking` оказался сильнее и дал `0.7721`.
- Pseudo-labeling дал самый большой прирост. Добавление `7_761` synthetic-примеров из `Lenta.ru` подняло качество до `0.8251` даже без MLM-инициализации.
- Комбинация улучшений тоже сработала: `entity MLM + synthetic` дала лучший итоговый результат `0.8328`.
- Полученные результаты сильные и хорошо согласуются с гипотезой задания: доменно близкая синтетика здесь полезнее, чем только MLM-предобучение на небольшом gold-корпусе.
- При этом называть результат глобально оптимальным нельзя. Он оптимален только среди уже проведённых контролируемых прогонов и в рамках ограничений текущей машины без GPU.
- Что можно было бы улучшить еще: прмиенить более сильный teacher для pseudo-labeling, фильтрацию synthetic-разметки по уверенности, curriculum `synthetic -> gold`, отдельный подбор learning rate и числа эпох для merged-train, а также переход на более крупный encoder при наличии GPU (но у меня его нет, а на коллабе конфликт с гуглом по лимитам места в облаке :О( ).